# 12.2 Spectral graph theory

The Laplacian turns connectivity into eigenvalues and eigenvectors.

The Laplacian $L=D-A$ turns cuts, smoothness, and diffusion into matrix questions. Spectral ideas explain GCN normalization and later structural graph biases.

Save a copy to Drive to edit.

In [ ]:

import math
import random

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

SEED = 12
rng = np.random.default_rng(SEED)
random.seed(SEED)


def adjacency_from_graph(G):
    nodes = list(G.nodes())
    return nx.to_numpy_array(G, nodelist=nodes, dtype=float)


def labels_from_graph(G):
    labels = []
    for node in G.nodes():
        club = G.nodes[node].get("club", "Mr. Hi")
        value = 1 if club == "Officer" else 0
        labels.append(value)
    return np.array(labels, dtype=int)


def degree_features(A):
    degree = A.sum(axis=1)
    max_degree = max(1.0, float(degree.max()))
    normalized_degree = degree / max_degree
    clustering = np.array([nx.clustering(nx.from_numpy_array(A), i) for i in range(A.shape[0])])
    return np.column_stack([normalized_degree, clustering])


def make_sbm_graph(sizes, p_in, p_out, seed, feature_noise):
    blocks = len(sizes)
    probs = np.full((blocks, blocks), p_out, dtype=float)
    np.fill_diagonal(probs, p_in)
    G = nx.stochastic_block_model(sizes, probs, seed=seed)
    y = []
    for block_id, size in enumerate(sizes):
        y.extend([block_id] * size)
    y = np.array(y, dtype=int)
    base = np.eye(blocks, dtype=float)[y]
    local_rng = np.random.default_rng(seed)
    noise = local_rng.normal(0.0, feature_noise, size=base.shape)
    X = base + noise
    return G, X, y


def make_graph_ladder(seed=SEED):
    ladder = []

    G1 = nx.Graph()
    G1.add_edges_from([(0, 1), (0, 2), (1, 2), (1, 3)])
    X1 = np.array([[1.0, 0.0], [1.0, 1.0], [0.0, 1.0], [0.0, 0.0]])
    y1 = np.array([0, 0, 1, 1], dtype=int)
    ladder.append({"rung": "D1", "name": "4-node toy", "G": G1, "A": adjacency_from_graph(G1), "X": X1, "y": y1})

    G2 = nx.karate_club_graph()
    A2 = adjacency_from_graph(G2)
    X2 = degree_features(A2)
    y2 = labels_from_graph(G2)
    ladder.append({"rung": "D2", "name": "karate club", "G": G2, "A": A2, "X": X2, "y": y2})

    G3, X3, y3 = make_sbm_graph([18, 18], 0.34, 0.04, seed + 3, 0.28)
    ladder.append({"rung": "D3", "name": "synthetic SBM with noise", "G": G3, "A": adjacency_from_graph(G3), "X": X3, "y": y3})

    G4, X4, y4 = make_sbm_graph([30, 30, 30], 0.28, 0.07, seed + 4, 0.35)
    ladder.append({"rung": "D4", "name": "larger denser SBM", "G": G4, "A": adjacency_from_graph(G4), "X": X4, "y": y4})

    G5, X5, y5 = make_sbm_graph([35, 35, 35], 0.18, 0.11, seed + 5, 0.55)
    hub_edges = []
    for node in range(10, 95, 7):
        hub_edges.append((0, node))
    G5.add_edges_from(hub_edges)
    A5 = adjacency_from_graph(G5)
    ladder.append({"rung": "D5", "name": "noisy hub-heavy SBM", "G": G5, "A": A5, "X": X5, "y": y5})

    return ladder


def centroid_predict(Z, y):
    classes = np.unique(y)
    centroids = []
    for cls in classes:
        centroids.append(Z[y == cls].mean(axis=0))
    centroids = np.vstack(centroids)
    distances = ((Z[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2)
    indices = distances.argmin(axis=1)
    return classes[indices]


def node_accuracy(pred, y):
    return float(np.mean(np.asarray(pred) == np.asarray(y)))


def conductance(A, labels):
    labels = np.asarray(labels)
    group = labels == labels[0]
    if group.all() or (not group.any()):
        return 1.0
    cut = float(A[np.ix_(group, ~group)].sum())
    volume_a = float(A[group].sum())
    volume_b = float(A[~group].sum())
    denom = max(1e-12, min(volume_a, volume_b))
    return cut / denom


def normalized_adjacency(A):
    A_tilde = A + np.eye(A.shape[0])
    degree = A_tilde.sum(axis=1)
    inv_sqrt = np.diag(1.0 / np.sqrt(np.maximum(degree, 1e-12)))
    return inv_sqrt @ A_tilde @ inv_sqrt


def plot_graph_panels(results, metric_name):
    cols = len(results)
    fig, axes = plt.subplots(1, cols, figsize=(4 * cols, 3))
    if cols == 1:
        axes = [axes]
    for ax, item in zip(axes, results):
        G = item["G"]
        colors = item["pred"]
        pos = nx.spring_layout(G, seed=SEED)
        nx.draw_networkx_nodes(G, pos, node_color=colors, cmap="viridis", node_size=80, ax=ax)
        nx.draw_networkx_edges(G, pos, alpha=0.25, width=0.7, ax=ax)
        ax.set_title(f'{item["rung"]}: {item["metric"]:.3f}')
        ax.axis("off")
    plt.show()

    plt.figure(figsize=(6, 3))
    xs = [item["rung"] for item in results]
    ys = [item["metric"] for item in results]
    plt.plot(xs, ys, marker="o")
    plt.ylabel(metric_name)
    plt.xlabel("ladder rung")
    plt.title(f"{metric_name} vs rung")
    plt.grid(True, alpha=0.3)
    plt.show()


## 3. The concept, built once (D1)

Build $L=D-A$, compute eigenvectors, measure smoothness by $x^\top Lx=\sum_{(i,j)\in E}(x_i-x_j)^2$, and compare the normalized Laplacian $I-D^{-1/2}AD^{-1/2}$. On the lesson path, the spectra are $0,0.586,2.000,3.414$ and $0,0.500,1.500,2.000$, while $x=[1,1,3,3]$ has energy $4.000$.

In [ ]:

def spectral_partition(A):
    degree = A.sum(axis=1)
    D = np.diag(degree)
    L = D - A
    eigvals, eigvecs = np.linalg.eigh(L)
    safe_degree = np.maximum(degree, 1e-12)
    inv_sqrt = np.diag(1.0 / np.sqrt(safe_degree))
    L_norm = np.eye(A.shape[0]) - inv_sqrt @ A @ inv_sqrt
    norm_eigvals, norm_eigvecs = np.linalg.eigh(L_norm)
    fiedler = eigvecs[:, 1]
    labels = (fiedler > np.median(fiedler)).astype(int)
    return L, eigvals, fiedler, labels, L_norm, norm_eigvals

G_path = nx.path_graph(4)
A_path = adjacency_from_graph(G_path)
L_path, eig_path, fiedler_path, labels_path, L_norm_path, norm_eig_path = spectral_partition(A_path)
x_path = np.array([1.0, 1.0, 3.0, 3.0])
energy_path = float(x_path @ L_path @ x_path)

assert np.allclose(np.round(eig_path, 3), [0.000, 0.586, 2.000, 3.414])
assert np.allclose(np.round(norm_eig_path, 3), [0.000, 0.500, 1.500, 2.000])
assert round(energy_path, 3) == 4.000
assert labels_path.tolist() in ([0, 0, 1, 1], [1, 1, 0, 0])


The Fiedler vector is a relaxed cut: signs can flip, but the two contiguous path halves remain the split. Normalization keeps high-degree nodes from dominating the eigensystem.

In [ ]:

print("L eigenvalues", np.round(eig_path, 3).tolist())
print("normalized eigenvalues", np.round(norm_eig_path, 3).tolist())
print("Fiedler split", labels_path.tolist())
print("energy", round(energy_path, 3))


## 4. The dataset ladder

The F11 graph ladder is inline and CPU-only: D1 is a hand-checkable four-node toy, D2 is the bundled NetworkX karate club graph, D3 is a noisy stochastic block model, D4 is a larger/denser synthetic graph, and D5 is a harder noisy hub-heavy graph.

In [ ]:

ladder = make_graph_ladder()

for item in ladder:
    A = item["A"]
    X = item["X"]
    y = item["y"]
    edge_count = int(A.sum() // 2)
    classes = np.unique(y).tolist()
    print(item["rung"], item["name"])
    print("  nodes", A.shape[0], "edges", edge_count, "features", X.shape, "classes", classes)
    print("  sample X", np.round(X[:3], 3).tolist())


In [ ]:

results = []

for item in ladder:
    L, eigvals, fiedler, labels, L_norm, norm_eigvals = spectral_partition(item["A"])
    metric = conductance(item["A"], labels)
    results.append({"rung": item["rung"], "G": item["G"], "pred": labels, "metric": metric})
    print(f'{item["rung"]} {item["name"]:24s} conductance={metric:.3f}')


In [ ]:

plot_graph_panels(results, "cut conductance")


## 7. Pitfall on the hardest rung: unnormalized spectra on hubs

Hub-heavy D5 can make the unnormalized Fiedler vector chase degree instead of communities. The normalized Laplacian fixes the scale by dividing through degrees.

In [ ]:

hard = ladder[-1]
A = hard["A"]
L, eigvals, fiedler, raw_labels, L_norm, norm_eigvals = spectral_partition(A)
norm_fiedler = np.linalg.eigh(L_norm)[1][:, 1]
norm_labels = (norm_fiedler > np.median(norm_fiedler)).astype(int)
raw_cond = conductance(A, raw_labels)
norm_cond = conductance(A, norm_labels)

print("raw conductance", round(raw_cond, 3))
print("normalized conductance", round(norm_cond, 3))
print("lower is better for this cut diagnostic")


## 8. Evaluate it + Practice

- Report the ladder metric (cut conductance) next to a no-skill majority-class or random partition baseline.
- Sanity-check D1 against the hand-computed assertions before trusting larger rungs.
- Ablation: use adjacency eigenvectors instead of Laplacian eigenvectors.
- Failure signals: tiny second eigenvalue but visually mixed communities, or hub-only splits.
- Keep all experiments CPU-only and seeded.

Practice prompts:
1. Change D3 noise and predict which metric moves first.
2. Add a small bridge between communities and inspect the visualization.
3. Replace the readout with a majority baseline and compare the curve.